# LQG vs Robust Controller



**Figures:** lqg_vs_robust.svg

In [ ]:
%matplotlib inline

In [ ]:
#!/usr/bin/env python3
"""
LQG vs Robust Controller: Fragility of LQG under unmodeled dynamics
====================================================================

Demonstrates that an LQG controller, designed for a nominal 2nd-order model,
becomes unstable when the true plant contains an unmodeled lightly-damped
resonance mode.  A robust output-feedback controller (designed with explicit
bandwidth limitations to account for model uncertainty) remains stable in
both cases.

Nominal model (used for LQG design):
    G_m(s) = 10 / [(s+1)(s+3)]

Perturbed plant (true plant with unmodeled flexible mode):
    G_p(s) = G_m(s) * omega_r^2 / (s^2 + 2*zeta_r*omega_r*s + omega_r^2)
    with omega_r = 12 rad/s, zeta_r = 0.05  (lightly-damped resonance)

LQG controller: aggressive observer-based design (crossover ~ 6 rad/s)
Robust controller: PI + lead + 2nd-order low-pass (crossover ~ 2.5 rad/s)
"""

import numpy as np
from scipy.linalg import solve_continuous_are
import control as ct
import matplotlib
import matplotlib.pyplot as plt

# =====================================================================
#  Matplotlib configuration
# =====================================================================
matplotlib.rcParams.update({
    "svg.fonttype":      "path",
    "mathtext.fontset":  "cm",
    "font.family":       "serif",
    "font.size":         10,
    "axes.labelsize":    11,
    "axes.titlesize":    12,
    "legend.fontsize":    9,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "lines.linewidth":   1.5,
    "axes.linewidth":    0.8,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linewidth":    0.5,
})

# MATLAB default colors
BLUE   = (0.0000, 0.4470, 0.7410)
ORANGE = (0.8500, 0.3250, 0.0980)
GRAY   = (0.5, 0.5, 0.5)

# =====================================================================
#  Plant definitions
# =====================================================================
s = ct.tf('s')

# Nominal model: G_m(s) = 10 / [(s+1)(s+3)]
Gm = ct.tf([10], [1, 4, 3])

# Unmodeled lightly-damped resonance
omega_r = 12.0        # resonance frequency  [rad/s]
zeta_r  = 0.05        # resonance damping    [-]
resonance = omega_r**2 / (s**2 + 2*zeta_r*omega_r*s + omega_r**2)

# Perturbed plant (nominal model * unmodeled mode)
Gp = Gm * resonance

# =====================================================================
#  LQG controller design  (aggressive, for the 2nd-order nominal model)
# =====================================================================
# State-space of G_m in controllable canonical form:
#   A = [0, 1; -3, -4],  B = [0; 1],  C = [10, 0]
Am = np.array([[0.0, 1.0],
               [-3.0, -4.0]])
Bm = np.array([[0.0], [1.0]])
Cm = np.array([[10.0, 0.0]])

# LQR weights  (aggressive state weighting, cheap control)
Q_lqr = np.diag([500.0, 10.0])
R_lqr = np.array([[0.01]])

P_lqr = solve_continuous_are(Am, Bm, Q_lqr, R_lqr)
K_lqr = (1.0 / R_lqr[0, 0]) * (Bm.T @ P_lqr)

# Kalman filter  (high model confidence, low measurement noise)
Q_kal = np.diag([100.0, 100.0])
R_kal = np.array([[0.01]])

Sigma_kal = solve_continuous_are(Am.T, Cm.T, Q_kal, R_kal)
L_kal = Sigma_kal @ Cm.T * (1.0 / R_kal[0, 0])

# Form LQG controller state-space (maps y -> u, negative gain built in)
#   xhat_dot = (A - B K - L C) xhat + L y
#   u        = -K xhat
A_k = Am - Bm @ K_lqr - L_kal @ Cm
Klqg_y_ss = ct.ss(A_k, L_kal, -K_lqr, np.zeros((1, 1)))
Klqg_y_tf = ct.minreal(ct.tf(Klqg_y_ss), verbose=False)

# Open-loop for standard negative-feedback analysis:
#   L(s) = -K_y(s) * G(s)   (flip sign: K_y has negative DC gain)
L_lqg_nom = ct.minreal(-Klqg_y_tf * Gm, verbose=False)

# Feedforward gain so that nominal closed-loop DC gain = 1
T_lqg_nom_raw = ct.feedback(L_lqg_nom, 1)
N_ff = 1.0 / ct.dcgain(T_lqg_nom_raw)

# Closed-loop transfer functions  (from r to y, with feedforward)
T_lqg_nom  = N_ff * ct.feedback(L_lqg_nom, 1)
L_lqg_pert = ct.minreal(-Klqg_y_tf * Gp, verbose=False)
T_lqg_pert = N_ff * ct.feedback(L_lqg_pert, 1)

# =====================================================================
#  Robust output-feedback controller  (H-infinity / mu-synthesis style)
#
#  Structure:  K_rob(s) = k_p * PI(s) * Lead(s) * LP(s)
#    PI:   (s + z_i) / s              — integral action
#    Lead: (s + z_l) / (s + p_l)      — phase boost at crossover
#    LP:   omega_f^2 / (s^2 + 2*zeta_f*omega_f*s + omega_f^2)
#                                      — high-frequency rolloff
#
#  Design rationale:
#    - Crossover at ~ 2.5 rad/s  (well below the uncertain resonance)
#    - 2nd-order LP ensures gain < 1 at the resonance frequency
#    - Phase margin > 60 degrees
# =====================================================================
kp   = 3.0
z_i  = 0.5     # PI zero
z_l  = 1.5     # lead zero
p_l  = 8.0     # lead pole
wf   = 5.0     # LP natural frequency
zf   = 0.7     # LP damping

K_rob = kp * (s + z_i)/s * (s + z_l)/(s + p_l) * wf**2/(s**2 + 2*zf*wf*s + wf**2)

T_rob_nom  = ct.feedback(ct.minreal(K_rob * Gm, verbose=False), 1)
T_rob_pert = ct.feedback(ct.minreal(K_rob * Gp, verbose=False), 1)

# =====================================================================
#  Diagnostics
# =====================================================================
gm_l, pm_l, _, wc_l = ct.margin(L_lqg_nom)
gm_r, pm_r, _, wc_r = ct.margin(ct.minreal(K_rob * Gm, verbose=False))

print("=" * 60)
print("LQG controller")
print(f"  K_lqr = [{K_lqr[0,0]:.2f}, {K_lqr[0,1]:.2f}]")
print(f"  L_kal = [{L_kal[0,0]:.2f}, {L_kal[1,0]:.2f}]")
print(f"  Crossover = {wc_l:.1f} rad/s,  PM = {pm_l:.1f} deg")
print(f"  Feedforward N = {N_ff:.4f}")
print()
print("Robust controller")
print(f"  Crossover = {wc_r:.1f} rad/s,  PM = {pm_r:.1f} deg")
print()

for label, T in [("LQG nominal",    T_lqg_nom),
                 ("LQG perturbed",   T_lqg_pert),
                 ("Robust nominal",  T_rob_nom),
                 ("Robust perturbed", T_rob_pert)]:
    poles = T.poles()
    maxre = max(poles.real)
    print(f"  {label:20s}: max Re(pole) = {maxre:+.3f}  "
          f"{'STABLE' if maxre < 0 else 'UNSTABLE'}")
print("=" * 60)

# =====================================================================
#  Simulate step responses
# =====================================================================
t = np.linspace(0, 8, 4000)

_, y_lqg_nom  = ct.step_response(T_lqg_nom,  t)
_, y_rob_nom  = ct.step_response(T_rob_nom,  t)
_, y_rob_pert = ct.step_response(T_rob_pert, t)

# Perturbed LQG diverges: simulate on a shorter window to avoid overflow
t_pert = np.linspace(0, 8, 4000)
try:
    _, y_lqg_pert = ct.step_response(T_lqg_pert, t_pert)
except Exception:
    y_lqg_pert = np.full_like(t_pert, np.nan)

# Clip for plotting
y_lqg_pert_clip = np.clip(y_lqg_pert, -6, 6)

# =====================================================================
#  Figure
# =====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.0))
fig.subplots_adjust(left=0.08, right=0.97, bottom=0.13, top=0.82,
                    wspace=0.28)

# Global title
fig.suptitle(
    r"LQG fragility under unmodeled dynamics"
    "\n"
    r"vs. robust output-feedback controller",
    fontsize=12, fontweight="bold", y=0.97,
)

# --- Left panel: nominal plant ---
ax1.plot(t, y_lqg_nom, color=BLUE,
         label=r"LQG ($\omega_c \!\approx\! 6\;\mathrm{rad/s}$)")
ax1.plot(t, y_rob_nom, color=ORANGE,
         label=r"Robust ($\omega_c \!\approx\! 2.5\;\mathrm{rad/s}$)")
ax1.axhline(1.0, color=GRAY, ls="--", lw=0.9, label="Reference")
ax1.set_xlabel(r"Time $t$ [s]")
ax1.set_ylabel(r"Output $y(t)$")
ax1.set_title("Nominal plant", pad=4)
ax1.set_xlim(0, 8)
ax1.set_ylim(-0.15, 1.35)
ax1.legend(loc="lower right", framealpha=0.92, edgecolor="0.8")

# --- Right panel: perturbed plant ---
ax2.plot(t_pert, y_lqg_pert_clip, color=BLUE,
         label=r"LQG $\longrightarrow$ unstable")
ax2.plot(t, y_rob_pert, color=ORANGE,
         label=r"Robust $\longrightarrow$ stable")
ax2.axhline(1.0, color=GRAY, ls="--", lw=0.9, label="Reference")

ax2.set_xlabel(r"Time $t$ [s]")
ax2.set_ylabel(r"Output $y(t)$")
ax2.set_title(
    r"Perturbed plant  (unmodeled mode: "
    r"$\omega_r\!=\!12\;\mathrm{rad/s}$, $\zeta_r\!=\!0.05$)",
    pad=4,
)
ax2.set_xlim(0, 8)
ax2.set_ylim(-6, 6)
ax2.legend(loc="upper right", framealpha=0.92, edgecolor="0.8")

# =====================================================================
#  Export
# =====================================================================
fig.savefig("lqg_vs_robust.svg", bbox_inches="tight")
fig.savefig("lqg_vs_robust.png", dpi=200, bbox_inches="tight")
print("\nFigure saved: lqg_vs_robust.svg / .png")
plt.close(fig)